# MW test campaign: density evolution & disk projections

*First run a **base Milky Way** reference simulation and examine it in detail, then launch the parameter sweep and compare variants against that baseline.*

## What this notebook covers

| Stage | Tool | Output |
| --- | --- | --- |
| Base reference | `milky_way_disk_halo` (default params) | Baseline ρ(r), Σ(R), projections, ΔE/E₀ |
| Grid expansion | `galacticsics.campaign` | List of `(label, GalaxyModel)` variants |
| Solve + sample | legacy `dbh` / `gendisk` / `genhalo` | Equilibrium ICs per grid point |
| Evolve | ntropy tiered leapfrog | Self-gravitating N-body test |
| Analysis | `ntropy.analysis` | ρ(r), Σ(R), face-on / edge-on maps |

We use **`GalaxyModel.milky_way_disk_halo`** with a tunable DBH Poisson grid (`DBH_COARSE_GRID`, `DBH_DR`, `DBH_NR`, `DBH_LMAX` in the config cell). Production defaults are `nr=20000`, `lmax=6`, `dr=0.02` kpc; laptop screening uses `coarse_grid=True` (caps at `nr=4000`, `lmax≤4` for reliable `diskdf`). Sampling and N-body evolution at N ≈ 2×10⁵ add substantial time beyond `dbh`.

**Prerequisites:** `make install-dev` (builds `legacy/bin`, installs both packages including Jupyter and pandas).

Figures and campaign outputs go to `notebooks/artifacts/campaign_walkthrough/` (gitignored).

## Campaign pipeline

```
campaigns/mw_grid.json  →  expand_grid  →  solve (dbh)  →  sample  →  evolve (ntropy)
                                              ↓                ↓            ↓
                                         dbh.dat           halo/disk    evolution/
```

Each grid point gets a content-addressed work directory `runs/{hash}/` with `model.json`, particle files, and a manifest row in `campaign_index.csv`.

**Units** (shared everywhere):

| Quantity | Unit |
| --- | --- |
| Length | kpc |
| Velocity | 100 km/s |
| Mass | 2.325×10⁹ M☉ |
| Time | 1 code unit ≈ 9.78 Myr ≈ **0.0098 Gyr** |
| G | 1 |

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from galacticsics.campaign.runner import run_campaign
from galacticsics.campaign.parallel_budget import (
    apply_parallel_env,
    format_parallel_budget,
    resolve_parallel_budget,
)
from galacticsics.campaign.spec import (
    GridSpec,
    expand_grid,
    format_dbh_grid_summary,
    load_grid_spec,
    preview_dbh_model,
)
from galacticsics.campaign.timestep_summary import (
    format_tiered_timestep_summary,
    summarize_tiered_timestep,
)
from galacticsics.models import GalaxyModel
from galacticsics.sampling.particles import ParticleSet
from ntropy.analysis.density import bin_spherical_density, compare_density_profiles
from ntropy.analysis.disk_density import (
    bin_midplane_surface_density,
    bin_plane_density,
    compare_surface_profiles,
)
from ntropy.analysis.tiered_diagnostics import (
    diagnostics_dataframe,
    plot_tiered_diagnostics,
)
from ntropy.integrations.galacticsics import (
    galacticsics_available,
    merge_galacticsics_components,
    require_galacticsics,
)
from ntropy.integrators.tiered import run_tiered_leapfrog
from ntropy.integrators.timestep import TimestepConfig
from ntropy.particle_types import TypeRegistry
from ntropy.particles import ParticleState
from ntropy.softening import total_energy
from ntropy.units import code_time_to_gyr

REPO = Path.cwd()
if not (REPO / "src" / "galacticsics").exists():
    REPO = REPO.parent  # running from notebooks/

ARTIFACTS = REPO / "notebooks" / "artifacts" / "campaign_walkthrough"
CAMPAIGN_ROOT = ARTIFACTS / "runs"
BASE_ROOT = ARTIFACTS / "base_mw"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Production-scale defaults (expect hours wall time for full notebook on a laptop)
N_DISK = 100_000
N_HALO = 100_000
END_TIME_GYR = 0.50
CHECKPOINT_GYR = [0.0, 0.05, 0.10]

# --- DBH Poisson grid (legacy solve) ---
# coarse_grid caps production milky_way (nr=20000) → nr≤4000, lmax≤4, dr≥0.05
DBH_COARSE_GRID = True
DBH_DR = None              # kpc; None = keep after coarse (or base default)
DBH_NR = None              # radial bins; None = keep after coarse
DBH_LMAX = None            # multipole order; None = keep after coarse (max 4)
# Presets:
#   laptop screening:  DBH_COARSE_GRID=True  (nr=4000, lmax=4, dr=0.05)
#   fast reference:    DBH_NR=800, DBH_LMAX=2, DBH_DR=0.1
#   production:        DBH_COARSE_GRID=False (nr=20000, lmax=6, dr=0.02)

# --- Tiered timestep tuning (ntropy evolve) ---
# dt_base: finest substep [code units]; particle bin b uses dt = dt_base * 2**b
DT_BASE = 0.25             # ~0.24 Myr at bin 0 (1 code unit ≈ 9.78 Myr)
TIMESTEP_ETA = 0.025         # accuracy η for dynamical-time criterion (try 0.01–0.05)
MAX_TIMESTEP_BIN = 6         # coarsest bin → dt_max = DT_BASE * 2**MAX_TIMESTEP_BIN
TIMESTEP_UPDATE_EVERY = 1    # re-assign bins every N fine substeps
INTEGRATOR_ORDER = 2         # 1 = symplectic Euler, 2 = velocity Verlet (default)

EVOLVE_TIMESTEP = TimestepConfig(
    eta=TIMESTEP_ETA,
    dt_base=DT_BASE,
    max_bin=MAX_TIMESTEP_BIN,
    update_every=TIMESTEP_UPDATE_EVERY,
)

# ntropy per-substep diagnostics (written to evolution/diagnostics.csv)
DIAGNOSTICS_EVERY = 1          # aggregate bin histograms + activity every fine step
PARTICLE_DUMP_EVERY = 50       # per-particle .npz every N steps (large at 200k N)

# Parallelism: use 75% of cores for MPI×OpenMP (leave headroom for OS / IDE)
CORE_FRACTION = 0.75
MPI_RANKS = 2                  # MPI ranks for ntropy evolve (serial fallback if unavailable)

# Live campaign logs: stages, legacy Fortran output, ntropy evolve lines, ETAs
VERBOSE_CAMPAIGN = True

# Re-run ntropy from ICs for intermediate checkpoints (very expensive at 200k particles).
REEVOLVE_FOR_CHECKPOINTS = False

# Set False to force re-solve after changing grid or particle counts
SKIP_DONE = False

_PARALLEL = resolve_parallel_budget(MPI_RANKS, core_fraction=CORE_FRACTION)
MPI_RANKS = _PARALLEL.mpi_ranks  # clamped to core budget
apply_parallel_env(_PARALLEL)

_TS = summarize_tiered_timestep(
    dt_base=DT_BASE,
    end_time_gyr=END_TIME_GYR,
    max_bin=MAX_TIMESTEP_BIN,
    eta=TIMESTEP_ETA,
    update_every=TIMESTEP_UPDATE_EVERY,
    integrator_order=INTEGRATOR_ORDER,
)

print(
    f"Target particles: {N_DISK:,} disk + {N_HALO:,} halo = {N_DISK + N_HALO:,} total"
)
print(f"Evolution: {END_TIME_GYR} Gyr | verbose: {VERBOSE_CAMPAIGN}")
print(format_tiered_timestep_summary(_TS))
print(format_dbh_grid_summary(
    preview_dbh_model(
        coarse=DBH_COARSE_GRID,
        dr=DBH_DR,
        nr=DBH_NR,
        lmax=DBH_LMAX,
    )
))
print(f"Parallel: {format_parallel_budget(_PARALLEL)}")
print(f"Re-evolve checkpoints: {REEVOLVE_FOR_CHECKPOINTS}")

if not galacticsics_available():
    raise RuntimeError(
        "GalactICS legacy binaries missing. Run `make install-dev` from the repo root."
    )
require_galacticsics()
print(f"Artifacts → {ARTIFACTS}")

Target particles: 100,000 disk + 100,000 halo = 200,000 total
Evolution: 0.5 Gyr | verbose: True
Tiered timestep:
  dt_base     : 0.25000 code (2.445 Myr, bin 0)
  dt_max      : 16.0000 code (156.48 Myr, bin 6)
  eta         : 0.025
  bin update  : every 1 fine substep(s)
  integrator  : order-2 leapfrog
  duration    : 0.5 Gyr = 51.12 code units in 204 fine substeps
DBH grid: dr=0.05 kpc, nr=4000, lmax=8 (r_max ≈ 200 kpc; cost scales ~ nr × lmax²)
Parallel: 8 cores → budget 6 (MPI 2 × OMP 3 = 6)
Re-evolve checkpoints: False
Artifacts → /home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough


## 1. Base Milky Way simulation (reference case)

Before sweeping parameters, run the **default** `GalaxyModel.milky_way_disk_halo()` end-to-end:

- `halo.v0 = 3.7`, `disk.mass = 17.0`, production grid (`nr=20000`, `lmax=6`)
- Same particle counts and evolution time as the sweep (`N_DISK`, `N_HALO`, `END_TIME_GYR`)

Outputs land in `base_mw/{hash}/`. Use this section to validate IC quality, inspect density structure, and establish a baseline for comparing grid variants later.

**While running:** `run_campaign` logs each stage with timestamps and ETAs, streams `dbh` / `gendisk` / `genhalo` stderr, and prints **ntropy evolve lines** during MPI tiered integration. Tune DBH grid (`DBH_*`) and ntropy timestep (`DT_BASE`, `TIMESTEP_ETA`, `MAX_TIMESTEP_BIN`) in the config cell.

In [2]:
registry = TypeRegistry.default_galaxy()


def load_merged_state(model_dir: Path) -> ParticleState:
    particles: dict[str, ParticleSet] = {}
    for name in ("halo", "bulge", "disk"):
        path = model_dir / name
        if path.exists():
            particles[name] = ParticleSet.from_ascii(path, component=name)
    return merge_galacticsics_components(particles, type_registry=registry)


def load_evolved_state(model_dir: Path, template: ParticleState) -> ParticleState | None:
    final_path = model_dir / "evolution" / "final.dat"
    if not final_path.exists():
        return None
    from ntropy.io.particles import read_particles_ascii

    data = read_particles_ascii(final_path)
    pos = np.column_stack([data["x"], data["y"], data["z"]])
    vel = np.column_stack([data["vx"], data["vy"], data["vz"]])
    return ParticleState.from_arrays(
        pos, vel, data["mass"], template.eps,
        type_id=template.type_id,
        tags=template.tags,
    )


def component_mask(state: ParticleState, name: str) -> np.ndarray:
    return state.tags == name


def halo_spherical_profile(state: ParticleState, *, n_bins: int = 24, r_max: float = 40.0):
    mask = component_mask(state, "halo")
    return bin_spherical_density(state.pos[mask], state.mass[mask], n_bins=n_bins, r_max=r_max)


def disk_surface_profile(state: ParticleState, *, n_bins: int = 20, r_max: float = 22.0):
    mask = component_mask(state, "disk")
    return bin_midplane_surface_density(state.pos[mask], state.mass[mask], n_bins=n_bins, r_max=r_max)


def disk_projections(
    state: ParticleState,
    *,
    half_extent: float = 20.0,
    n_bins: int = 96,
    z_slice: float | None = 0.3,
):
    mask = component_mask(state, "disk")
    pos, mass = state.pos[mask], state.mass[mask]
    face_on = bin_plane_density(pos, mass, axes=(0, 1), n_bins=n_bins, half_extent=half_extent)
    z_filter = np.abs(pos[:, 2]) < z_slice if z_slice is not None else None
    edge_on = bin_plane_density(
        pos, mass, axes=(0, 2), n_bins=n_bins, half_extent=half_extent, z_filter=z_filter
    )
    return face_on, edge_on


def plot_density_map(ax, density_map, *, title: str, cmap: str = "magma"):
    extent = [
        density_map.x_edges[0], density_map.x_edges[-1],
        density_map.y_edges[0], density_map.y_edges[-1],
    ]
    data = np.log10(np.maximum(density_map.density.T, 1e-30))
    im = ax.imshow(data, origin="lower", extent=extent, aspect="equal", cmap=cmap)
    ax.set_title(title)
    return im


def model_summary_table(model: GalaxyModel) -> pd.DataFrame:
    rows = [
        ("halo.v0", model.halo.v0 if model.halo else None),
        ("halo.a [kpc]", model.halo.a if model.halo else None),
        ("halo.r_outer [kpc]", model.halo.r_outer if model.halo else None),
        ("disk.mass", model.disk.mass if model.disk else None),
        ("disk.scale_length [kpc]", model.disk.scale_length if model.disk else None),
        ("disk.scale_height [kpc]", model.disk.scale_height if model.disk else None),
        ("grid.nr", model.grid.nr),
        ("grid.lmax", model.grid.lmax),
        ("grid.dr [kpc]", model.grid.dr),
    ]
    return pd.DataFrame(rows, columns=["parameter", "value"])


def load_run_diagnostics(work_dir: Path):
    """Load tiered integrator diagnostics from ``work_dir/evolution/``."""
    evo_dir = work_dir / "evolution"
    return diagnostics_dataframe(evo_dir), evo_dir

### 1a. Run solve → sample → evolve for the base model

In [3]:
BASE_SPEC = GridSpec(
    name="base_mw",
    base="milky_way_disk_halo",
    axes={},
    omit_components=[[]],
    coarse_grid=DBH_COARSE_GRID,
    grid_dr=DBH_DR,
    grid_nr=DBH_NR,
    grid_lmax=DBH_LMAX,
)

base_label, BASE_MODEL = expand_grid(BASE_SPEC)[0]
model_summary_table(BASE_MODEL)

base_manifest = run_campaign(
    BASE_SPEC,
    BASE_ROOT,
    stages=["solve", "sample", "evolve"],
    n_disk=N_DISK,
    n_halo=N_HALO,
    n_bulge=0,
    end_time_gyr=END_TIME_GYR,
    dt_base=DT_BASE,
    timestep_eta=TIMESTEP_ETA,
    max_timestep_bin=MAX_TIMESTEP_BIN,
    timestep_update_every=TIMESTEP_UPDATE_EVERY,
    integrator_order=INTEGRATOR_ORDER,
    diagnostics_every=DIAGNOSTICS_EVERY,
    particle_dump_every=PARTICLE_DUMP_EVERY,
    skip_done=SKIP_DONE,
    verbose=VERBOSE_CAMPAIGN,
    mpi_ranks=MPI_RANKS,
    core_fraction=CORE_FRACTION,
)

base_row = base_manifest.rows[0]
BASE_WORK_DIR = Path(base_row["path"])
print(f"Base model: {base_label}")
print(f"Work dir: {BASE_WORK_DIR}")
pd.DataFrame([base_row])

[22:04:36] ========================================================================
[22:04:36] Campaign 'base_mw' → /home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw
[22:04:36] Stages: solve, sample, evolve | models: 1
[22:04:36]   particles: 100,000 disk + 100,000 halo
[22:04:36]   end_time_gyr: 0.5
[22:04:36]   dt_base: 0.25
[22:04:36]   mpi_ranks: 2
[22:04:36]   core_fraction: 0.75
[22:04:36]   diagnostics_every: 1
[22:04:36]   skip_done: False
[22:04:36] ========================================================================
[22:04:36] ------------------------------------------------------------------------
[22:04:36] Model [1/1] full  hash=b0a9d509e8e7…
  dir: /home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw/b0a9d509e8e7


full:   0%|          | 0/3 [00:00<?, ?stage/s]

[22:04:36] ▶ solve — dbh (nr=4000, lmax=8)
[22:04:36]   legacy: dbh  (cwd=/home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw/b0a9d509e8e7)
[22:04:36]   ✓ solve finished in 0s | rtidal=0.0008097 | solve_seconds=0.4741
[22:04:36] ▶ sample — n_disk=100,000, n_halo=100,000, n_bulge=0
[22:04:36]   legacy: gendisk (+ diskdf if needed)  (cwd=/home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw/b0a9d509e8e7)
[22:04:36]   legacy: genhalo  (cwd=/home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw/b0a9d509e8e7)


   1.76380002       7.61409989E-03
 Harmonics up to l=   8
 Harmonics up to l=   8
 Central velocity dispersion, scale length of sigR**2?
 number of radial steps for correction fns (min. 6)?
 number of iterations?
 wrote a table of frequencies in file omekap.dat.
 Toomre Q =   0.864805579      at R = 2.5 R_d
 finding a not-a-number in splintd
              NaN              NaN
              NaN
              NaN
              NaN
              NaN
Note: The following floating-point exceptions are signalling: IEEE_INVALID_FLAG IEEE_DIVIDE_BY_ZERO
Enter the number of particles [10000]: Enter negative integer seed [-123]: Center the simulation (0=no,1=yes) [1]: At line 17 of file readdiskdf.f (unit = 17, file = 'cordbh.dat')
Fortran runtime error: End of file


LegacyRunError: gendisk failed (code 2) in /home/jbauer/GalactICSIsoWithGas/notebooks/artifacts/campaign_walkthrough/base_mw/b0a9d509e8e7
stderr:


### 1b. Detailed base-case diagnostics

Particle counts, halo ρ(r) and disk Σ(R) at t=0 vs t=`END_TIME_GYR`, disk face-on / edge-on projections, and energy drift. These plots are the reference against which grid variants should be compared.

In [ ]:
base_initial = load_merged_state(BASE_WORK_DIR)
base_final = load_evolved_state(BASE_WORK_DIR, base_initial)
if base_final is None:
    raise FileNotFoundError("Base evolve stage did not write evolution/final.dat")

print(f"N = {base_initial.n:,} particles")
for label in registry.types:
    mask = base_initial.tags == label
    if np.any(mask):
        print(f"  {label:5s}: {mask.sum():6,d}")

e0 = total_energy(base_initial.pos, base_initial.vel, base_initial.mass, base_initial.eps)
ef = total_energy(base_final.pos, base_final.vel, base_final.mass, base_final.eps)
base_dE = abs(ef - e0) / max(abs(e0), 1e-30)
print(f"|ΔE/E₀| after {END_TIME_GYR} Gyr: {base_dE:.3e}")
if "dE_over_E0" in base_row:
    print(f"Campaign-reported |ΔE/E₀|: {base_row['dE_over_E0']:.3e}")

# --- profiles: initial vs final ---
halo_i = halo_spherical_profile(base_initial)
halo_f = halo_spherical_profile(base_final)
disk_i = disk_surface_profile(base_initial)
disk_f = disk_surface_profile(base_final)
halo_drift = compare_density_profiles(halo_i, halo_f, min_count=20)
disk_drift = compare_surface_profiles(disk_i, disk_f, min_count=20)
print(f"Max relative halo ρ drift: {halo_drift:.3f}")
print(f"Max relative disk Σ drift: {disk_drift:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, prof_i, prof_f, title, logy in (
    (axes[0, 0], halo_i, halo_f, "Halo ρ(r)", True),
    (axes[0, 1], disk_i, disk_f, "Disk Σ(R)", False),
):
    vi, vf = prof_i.counts > 0, prof_f.counts > 0
    if logy:
        ax.loglog(prof_i.r_mid[vi], prof_i.rho[vi], "C0-", label="t=0", ms=3)
        ax.loglog(prof_f.r_mid[vf], prof_f.rho[vf], "C1--", label=f"t={END_TIME_GYR} Gyr", ms=3)
        ax.set_ylabel("ρ [M_unit / kpc³]")
    else:
        ax.semilogy(prof_i.r_mid[vi], prof_i.sigma[vi], "C0-", label="t=0", ms=3)
        ax.semilogy(prof_f.r_mid[vf], prof_f.sigma[vf], "C1--", label=f"t={END_TIME_GYR} Gyr", ms=3)
        ax.set_ylabel("Σ [M_unit / kpc²]")
    ax.set_xlabel("r or R [kpc]")
    ax.set_title(title)
    ax.legend(fontsize=8)

face_i, edge_i = disk_projections(base_initial)
face_f, edge_f = disk_projections(base_final)
im0 = plot_density_map(axes[1, 0], face_i, title="Face-on t=0")
im1 = plot_density_map(axes[1, 1], face_f, title=f"Face-on t={END_TIME_GYR} Gyr")
axes[1, 0].set_xlabel("x [kpc]"); axes[1, 0].set_ylabel("y [kpc]")
axes[1, 1].set_xlabel("x [kpc]"); axes[1, 1].set_ylabel("y [kpc]")
fig.colorbar(im0, ax=axes[1, :].tolist(), fraction=0.03, label="log₁₀ Σ")

fig.suptitle("Base Milky Way — density structure", y=1.01)
fig.tight_layout()
base_profile_path = ARTIFACTS / "base_density_diagnostics.png"
fig.savefig(base_profile_path, dpi=150, bbox_inches="tight")
plt.show()

# --- edge-on comparison ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
plot_density_map(axes[0], edge_i, title="Edge-on t=0 (|z| < 0.3 kpc)")
plot_density_map(axes[1], edge_f, title=f"Edge-on t={END_TIME_GYR} Gyr")
for ax in axes:
    ax.set_xlabel("x [kpc]")
axes[0].set_ylabel("z [kpc]")
fig.tight_layout()
base_edge_path = ARTIFACTS / "base_disk_edge_on.png"
fig.savefig(base_edge_path, dpi=150)
plt.show()
print(f"Saved {base_profile_path.name}, {base_edge_path.name}")

### 1c. ntropy bin / activity diagnostics (per fine step)

Each tiered leapfrog run writes:

- `evolution/diagnostics.csv` — energy drift, active fraction, global bin histogram, per-type mean bin **every fine substep**
- `evolution/particles/step_XXXXXX.npz` — per-particle `timestep_bin`, `type_id`, `accel_mag`, positions (every `PARTICLE_DUMP_EVERY` steps)

Re-run with `SKIP_DONE = False` if these files are missing from an older evolve stage.

In [ ]:
base_diag_dir = BASE_WORK_DIR / "evolution"
if not (base_diag_dir / "diagnostics.csv").exists():
    raise FileNotFoundError(
        f"Missing {base_diag_dir / 'diagnostics.csv'} — re-run base evolve with SKIP_DONE=False"
    )

base_diag_df, _ = load_run_diagnostics(BASE_WORK_DIR)
print(f"{len(base_diag_df)} diagnostic records, t = 0 … {base_diag_df.t_gyr.iloc[-1]:.4f} Gyr")

fig = plot_tiered_diagnostics(base_diag_df)
diag_fig_path = ARTIFACTS / "base_ntropy_diagnostics.png"
fig.savefig(diag_fig_path, dpi=150, bbox_inches="tight")
plt.show()

particle_dir = base_diag_dir / "particles"
npz_files = sorted(particle_dir.glob("step_*.npz"))
if npz_files:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, path in zip(axes, [npz_files[0], npz_files[-1]]):
        data = np.load(path)
        bins = data["timestep_bin"]
        type_id = data["type_id"]
        for tid, label, color in [(1, "halo", "C0"), (3, "disk", "C1")]:
            mask = type_id == tid
            if np.any(mask):
                ax.hist(bins[mask], bins=np.arange(-0.5, 8.5, 1), alpha=0.6, label=label, color=color)
        ax.set_xlabel("timestep bin")
        ax.set_ylabel("N particles")
        ax.set_title(path.name)
        ax.legend(fontsize=8)
    fig.suptitle("Per-particle timestep bins (sampled steps)")
    fig.tight_layout()
    plt.show()
else:
    print("No particle bin dumps in", particle_dir)
print(f"Saved {diag_fig_path.name}")

## 2. Define and inspect the parameter sweep

Mini grid (two halo circular velocities) based on `campaigns/mw_grid.json`, using the same DBH grid knobs as the base case (`DBH_COARSE_GRID`, `DBH_NR`, etc.).

Compare sweep outputs in later sections against the **base reference** in `BASE_WORK_DIR`.

In [5]:
# Production DBH + mini parameter sweep (2 models). See campaigns/mw_grid.json for the full factorial grid.
NOTEBOOK_SPEC = GridSpec(
    name="notebook_production",
    base="milky_way_disk_halo",
    axes={"halo.v0": [3.5, 3.7]},
    omit_components=[[]],
    coarse_grid=DBH_COARSE_GRID,
    grid_dr=DBH_DR,
    grid_nr=DBH_NR,
    grid_lmax=DBH_LMAX,
    max_points=10,
)

grid_models = expand_grid(NOTEBOOK_SPEC)
rows = []
for label, model in grid_models:
    rows.append(
        {
            "label": label,
            "halo_v0": model.halo.v0 if model.halo else None,
            "disk_mass": model.disk.mass if model.disk else None,
            "grid_nr": model.grid.nr,
            "grid_lmax": model.grid.lmax,
            "grid_dr": model.grid.dr,
        }
    )
grid_df = pd.DataFrame(rows)
grid_df

,label,halo_v0,disk_mass,grid_nr,grid_lmax
0,halo_v0=3.5__full,3.5,17.0,800,2
1,halo_v0=3.7__full,3.7,17.0,800,2


## 3. Run the parameter sweep (solve → sample → evolve)

After validating the base case, run the mini halo-velocity sweep with the same pipeline as section 1:

1. **solve** — full `dbh` (`nr=20000`, `lmax=6`)
2. **sample** — `gendisk` / `genhalo` at `N_DISK` + `N_HALO`
3. **evolve** — ntropy tiered leapfrog to `END_TIME_GYR` (`MPI_RANKS=2` by default, `CORE_FRACTION=0.75`)

Set `SKIP_DONE = False` or delete `CAMPAIGN_ROOT` to re-run sweep points after config changes. The manifest (`campaign_index.csv`) records wall times and |ΔE/E₀|. With `VERBOSE_CAMPAIGN = True` (default), `run_campaign` prints timestamped stage logs, streams legacy Fortran stderr, and tails ntropy evolve lines (also written to `evolution/diagnostics.progress.jsonl`).

In [ ]:
manifest = run_campaign(
    NOTEBOOK_SPEC,
    CAMPAIGN_ROOT,
    stages=["solve", "sample", "evolve"],
    n_disk=N_DISK,
    n_halo=N_HALO,
    n_bulge=0,
    end_time_gyr=END_TIME_GYR,
    dt_base=DT_BASE,
    timestep_eta=TIMESTEP_ETA,
    max_timestep_bin=MAX_TIMESTEP_BIN,
    timestep_update_every=TIMESTEP_UPDATE_EVERY,
    integrator_order=INTEGRATOR_ORDER,
    diagnostics_every=DIAGNOSTICS_EVERY,
    particle_dump_every=PARTICLE_DUMP_EVERY,
    skip_done=SKIP_DONE,
    verbose=VERBOSE_CAMPAIGN,
    mpi_ranks=MPI_RANKS,
    core_fraction=CORE_FRACTION,
)

manifest_df = pd.DataFrame(manifest.rows)
manifest_df

## 4. Load a sweep point for detailed analysis

Pick one grid point (default: first row) and reload its ICs and evolved state. The analysis below mirrors section 1b for a single sweep variant.

In [ ]:
EXAMPLE_IDX = 0
example = manifest_df.iloc[EXAMPLE_IDX]
work_dir = Path(example["path"])
print(f"Sweep model: {example['label']}\nWork dir: {work_dir}")
print(f"Base reference: {BASE_WORK_DIR}")

initial_state = load_merged_state(work_dir)
final_state = load_evolved_state(work_dir, initial_state)

print(f"N = {initial_state.n:,} particles")
for label in registry.types:
    mask = initial_state.tags == label
    if np.any(mask):
        print(f"  {label:5s}: {mask.sum():6,d}")

### 4a. Sweep point — initial density and disk projections (t = 0)

In [ ]:
halo_init = halo_spherical_profile(initial_state)
disk_init = disk_surface_profile(initial_state)
face_init, edge_init = disk_projections(initial_state)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].loglog(halo_init.r_mid[halo_init.counts > 0], halo_init.rho[halo_init.counts > 0], "o-", ms=4)
axes[0].set_xlabel("r [kpc]")
axes[0].set_ylabel("ρ [M_unit / kpc³]")
axes[0].set_title(f"Halo ρ(r) — {example['label']} (t=0)")

axes[1].semilogy(disk_init.r_mid[disk_init.counts > 0], disk_init.sigma[disk_init.counts > 0], "o-", ms=4)
axes[1].set_xlabel("R [kpc]")
axes[1].set_ylabel("Σ [M_unit / kpc²]")
axes[1].set_title("Disk Σ(R) — t=0")

im = plot_density_map(axes[2], face_init, title="Disk face-on (log Σ)")
axes[2].set_xlabel("x [kpc]")
axes[2].set_ylabel("y [kpc]")
fig.colorbar(im, ax=axes[2], fraction=0.046, label="log₁₀ Σ")
fig.tight_layout()
init_path = ARTIFACTS / "sweep_initial_density.png"
fig.savefig(init_path, dpi=150)
plt.show()
print(f"Saved {init_path.name}")

### 4b. Edge-on disk projections (sweep point, t = 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = plot_density_map(axes[0], face_init, title="Face-on (x–y)")
axes[0].set_xlabel("x [kpc]")
axes[0].set_ylabel("y [kpc]")
im1 = plot_density_map(axes[1], edge_init, title="Edge-on (x–z, |z| < 0.3 kpc)")
axes[1].set_xlabel("x [kpc]")
axes[1].set_ylabel("z [kpc]")
fig.colorbar(im0, ax=axes[0], fraction=0.046, label="log₁₀ Σ")
fig.colorbar(im1, ax=axes[1], fraction=0.046, label="log₁₀ Σ")
fig.tight_layout()
proj_path = ARTIFACTS / "sweep_disk_projections_initial.png"
fig.savefig(proj_path, dpi=150)
plt.show()
print(f"Saved {proj_path.name}")

In [ ]:
# Quick comparison: sweep point vs base reference (final disk Σ)
fig, ax = plt.subplots(figsize=(7, 4))
for st, label, style in (
    (base_final, "base MW", "k-"),
    (final_state, example["label"], "C1--"),
):
    if st is None:
        continue
    prof = disk_surface_profile(st)
    valid = prof.counts > 10
    ax.semilogy(prof.r_mid[valid], prof.sigma[valid], style, label=label, ms=3)
ax.set_xlabel("R [kpc]")
ax.set_ylabel("Σ [M_unit / kpc²]")
ax.set_title(f"Final disk Σ(R) vs base ({END_TIME_GYR:.2f} Gyr)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 4c. Sweep point — ntropy diagnostics

Same per-substep bin / activity traces for the selected sweep variant (`work_dir`).

In [ ]:
sweep_diag_df, sweep_diag_dir = load_run_diagnostics(work_dir)
fig = plot_tiered_diagnostics(sweep_diag_df)
fig.suptitle(f"ntropy diagnostics — {example['label']}", y=1.02)
fig.savefig(ARTIFACTS / "sweep_ntropy_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Time-resolved evolution with checkpoints

By default (`REEVOLVE_FOR_CHECKPOINTS = False`) we compare **initial ICs** vs the campaign's **`evolution/final.dat`** — avoiding a second 200k-particle integration.

Set `REEVOLVE_FOR_CHECKPOINTS = True` to re-integrate from ICs in segments at `CHECKPOINT_GYR` (same tiered leapfrog as the campaign, but much slower).

In [ ]:
def make_accel_fn(state: ParticleState, registry: TypeRegistry):
    from ntropy.forces.bhtree_c import compute_forces_bh_c, extension_available
    from ntropy.forces.bhtree import compute_forces_bh

    use_c = extension_available()

    def accel(pos: np.ndarray) -> np.ndarray:
        if use_c:
            return compute_forces_bh_c(pos, state.mass, state.eps, theta=0.5)
        return compute_forces_bh(pos, state.mass, state.eps, theta=0.5)

    return accel


def evolve_with_checkpoints(
    state0: ParticleState,
    checkpoint_gyr: list[float],
    *,
    registry: TypeRegistry,
    ts_config: TimestepConfig | None = None,
) -> tuple[list[tuple[float, ParticleState]], list[float]]:
    ts_config = ts_config or EVOLVE_TIMESTEP
    state = state0.copy()
    accel_fn = make_accel_fn(state, registry)
    checkpoints: list[tuple[float, ParticleState]] = [(checkpoint_gyr[0], state0.copy())]
    energies: list[float] = [total_energy(state.pos, state.vel, state.mass, state.eps)]

    for t_prev, t_end in zip(checkpoint_gyr[:-1], checkpoint_gyr[1:]):
        dt_gyr = t_end - t_prev
        state, seg_energies, _diag = run_tiered_leapfrog(
            state, registry, accel_fn,
            ts_config=ts_config, end_time_gyr=dt_gyr, order=INTEGRATOR_ORDER, energy_every=0,
            diagnostics_every=0,
        )
        if seg_energies:
            energies.extend(seg_energies[1:])
        checkpoints.append((t_end, state.copy()))

    return checkpoints, energies


ts_config = EVOLVE_TIMESTEP

if REEVOLVE_FOR_CHECKPOINTS:
    checkpoints, checkpoint_energies = evolve_with_checkpoints(
        initial_state, CHECKPOINT_GYR, registry=registry, ts_config=ts_config,
    )
else:
    final_evolved = load_evolved_state(work_dir, initial_state)
    if final_evolved is None:
        raise FileNotFoundError(
            f"No {work_dir / 'evolution' / 'final.dat'} — run campaign evolve stage first"
        )
    checkpoints = [(0.0, initial_state.copy()), (END_TIME_GYR, final_evolved)]
    checkpoint_energies = [
        total_energy(initial_state.pos, initial_state.vel, initial_state.mass, initial_state.eps),
        total_energy(final_evolved.pos, final_evolved.vel, final_evolved.mass, final_evolved.eps),
    ]

print(f"{len(checkpoints)} time slices: ", ", ".join(f"{t:.3f} Gyr" for t, _ in checkpoints))

### 5a. Density profile drift vs time

For a good equilibrium IC, spherical halo ρ(r) and disk Σ(R) should be stable over a few dynamical times — large drift signals poor IC quality, timestep issues, or violence from disk+halo coupling.

In [ ]:
ref_halo = halo_spherical_profile(checkpoints[0][1])
ref_disk = disk_surface_profile(checkpoints[0][1])

halo_drifts, disk_drifts, times = [], [], []
for t_gyr, st in checkpoints[1:]:
    hp = halo_spherical_profile(st)
    dp = disk_surface_profile(st)
    halo_drifts.append(compare_density_profiles(ref_halo, hp, min_count=5))
    disk_drifts.append(compare_surface_profiles(ref_disk, dp, min_count=5))
    times.append(t_gyr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, profiles, title, ylabel in (
    (axes[0], [halo_spherical_profile(st) for _, st in checkpoints], "Halo ρ(r)", "ρ"),
    (axes[1], [disk_surface_profile(st) for _, st in checkpoints], "Disk Σ(R)", "Σ"),
):
    cmap = plt.cm.viridis(np.linspace(0.15, 0.95, len(checkpoints)))
    for (t_gyr, _), prof, color in zip(checkpoints, profiles, cmap):
        valid = prof.counts > 0
        if title.startswith("Halo"):
            ax.loglog(prof.r_mid[valid], prof.rho[valid], color=color, alpha=0.85, label=f"{t_gyr:.3f} Gyr")
        else:
            ax.semilogy(prof.r_mid[valid], prof.sigma[valid], color=color, alpha=0.85, label=f"{t_gyr:.3f} Gyr")
    ax.set_xlabel("r or R [kpc]")
    ax.set_ylabel(f"{ylabel} [code units]")
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.tight_layout()
drift_path = ARTIFACTS / "density_evolution.png"
fig.savefig(drift_path, dpi=150)
plt.show()

e0 = max(abs(checkpoint_energies[0]), 1e-30)
dE = [abs(e - checkpoint_energies[0]) / e0 for e in checkpoint_energies]
print("Max relative halo ρ drift:", max(halo_drifts) if halo_drifts else 0.0)
print("Max relative disk Σ drift:", max(disk_drifts) if disk_drifts else 0.0)
print(f"|ΔE/E₀| at end: {dE[-1]:.3e}")

### 5b. Disk projections through time

Face-on maps show whether the disk stays axisymmetric; edge-on maps track vertical heating and warp growth.

In [ ]:
n_times = len(checkpoints)
fig, axes = plt.subplots(2, n_times, figsize=(3.2 * n_times, 6.5), squeeze=False)

for col, (t_gyr, st) in enumerate(checkpoints):
    face, edge = disk_projections(st)
    im0 = plot_density_map(axes[0, col], face, title=f"Face-on t={t_gyr:.3f} Gyr")
    im1 = plot_density_map(axes[1, col], edge, title=f"Edge-on t={t_gyr:.3f} Gyr")
    axes[0, col].set_xlabel("x [kpc]")
    axes[1, col].set_xlabel("x [kpc]")
    if col == 0:
        axes[0, col].set_ylabel("y [kpc]")
        axes[1, col].set_ylabel("z [kpc]")

fig.colorbar(im0, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, label="log₁₀ Σ")
fig.suptitle("Disk surface-density projections", y=1.02)
fig.tight_layout()
movie_path = ARTIFACTS / "disk_projections_evolution.png"
fig.savefig(movie_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {movie_path.name}")

## 6. Compare sweep points (and base reference)

Overlay final disk Σ(R) for each sweep variant against the **base Milky Way** run from section 1.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

base_prof = disk_surface_profile(base_final)
vb = base_prof.counts > 10
ax.semilogy(base_prof.r_mid[vb], base_prof.sigma[vb], "k-", lw=2, label="base MW")

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(manifest_df)))
for color, row in zip(colors, manifest_df.itertuples()):
    model_dir = Path(row.path)
    template = load_merged_state(model_dir)
    st = load_evolved_state(model_dir, template) or template
    prof = disk_surface_profile(st)
    valid = prof.counts > 10
    ax.semilogy(
        prof.r_mid[valid], prof.sigma[valid], color=color,
        label=f"{row.label}  ΔE/E₀={getattr(row, 'dE_over_E0', float('nan')):.2e}",
    )

ax.set_xlabel("R [kpc]")
ax.set_ylabel("Σ [M_unit / kpc²]")
ax.set_title(f"Final disk Σ(R) — base + sweep ({END_TIME_GYR:.2f} Gyr)")
ax.legend(fontsize=8)
fig.tight_layout()
grid_path = ARTIFACTS / "campaign_grid_comparison.png"
fig.savefig(grid_path, dpi=150)
plt.show()
print(f"Saved {grid_path.name}")

## Next steps

- Set `REEVOLVE_FOR_CHECKPOINTS = True` and denser `CHECKPOINT_GYR` for full time-resolved movies (slow at 200k particles)
- Extend `END_TIME_GYR` toward 1 Gyr for long-term stability tests
- Run the full factorial grid in `campaigns/mw_grid.json` (set `base: milky_way_disk_halo`, `coarse_grid: false`)
- Export particle tokens: `examples/export_particle_features.py`
- See [ml_representation_roadmap.md](../docs/ml_representation_roadmap.md) for transformer-based encoders